# Example 9.2 / 2: Read Compound Keys

Companion notebook for **9.2 Reading Tables** in the AMPL lab Documentation, section "Read Compound Keys". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Read Compound Keys

A long table has one row per product-period pair. Here `market.csv` also contains a row for
`wire`, a product the model does not know. Two declarations read it: `Market` into a parameter
indexed over `PROD` and `PERIOD`, and `Pairs` into a set of pairs filled from the keys:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                         # products
set PERIOD;                       # planning periods
param market {PROD, PERIOD} >= 0; # sales limit by product and period

set PAIRS dimen 2;                # product-period pairs present in the file
param listed {PAIRS};             # the file's market value for each pair

table Market IN "amplcsv" "market.csv":
    [product, period], market;
table Pairs IN "amplcsv" "market.csv":
    PAIRS <- [product, period], listed ~ market;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
with open("market.csv", "w") as f:
    f.write("product,period,market\n"
            "bands,1,6000\nbands,2,5000\n"
            "coils,1,4000\ncoils,2,4500\n"
            "plate,1,3500\nplate,2,3000\n"
            "wire,1,100\n")
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.set["PERIOD"] = [1, 2]

In [5]:
# This step fails on purpose, as on the page; the error is printed so the notebook goes on
try:
    # run.py
    ampl.eval("load amplcsv.dll;")
    ampl.read_table("Pairs")
    print(len(ampl.set["PAIRS"].to_list()), "pairs read")
    ampl.read_table("Market")
    print("Market read without error")
    print(ampl.param["market"].to_dict())
except Exception as error:
    print(f"{type(error).__name__}: {error}")

7 pairs read
Market read without error
RuntimeError: Error executing "display" command:
error processing param market:
	invalid subscript market['wire',1] discarded.


### Pure AMPL: data.dat and model.run

`Pairs` accepts all seven rows, because its keys define the set. `Market` needs the members of
`PROD` and `PERIOD` to exist already. The `read table` itself succeeds even though `wire` is not
in `PROD`: AMPL stores the values and checks the subscripts only when the parameter is first
used, here by `to_dict()`, which fails on `market['wire',1]`. Force that check right after the
read by displaying or otherwise using the loaded parameters, so a bad row is reported before
the solve.

Pure AMPL, with the same `model.mod` and `market.csv`:

In [6]:
%%writefile data.dat
set PROD := bands coils plate;
set PERIOD := 1 2;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
load amplcsv.dll;
read table Pairs;
display PAIRS;
read table Market;

set PAIRS :=
(bands,1)   (coils,1)   (plate,1)   (wire,1)
(bands,2)   (coils,2)   (plate,2);



In [9]:
# This step fails on purpose, as on the page; the error is printed so the notebook goes on
try:
    ampl.eval(r"""
# model.run
display market;
""")
except Exception:
    pass  # AMPL has printed the error above

[ERROR] 
	Error executing "display" command:
	error processing param market:
		invalid subscript market['wire',1] discarded.


market :=
bands 1   6000
bands 2   5000
coils 1   4000
coils 2   4500
plate 1   3500
plate 2   3000
;

